<a href="https://colab.research.google.com/github/Krispavnn555/NASSCOM_FDP/blob/main/DAY11_U23_Cloud_Storage_Databases_Part2_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# U23 — Cloud Storage & Databases (finish): Lab

### Real-world brief: making a plant-telemetry database fast & scalable

An industrial plant streams **200,000 sensor readings** from 30 machines into a database, and the analytics queries are getting slow. In this Part-2 lab you'll apply the scaling and performance techniques from the deck **hands-on** with SQLite + Python: **indexing & query plans**, a **cache-aside** layer, **read replicas** with lag, **sharding** across nodes, a **materialized view**, and basic **security** (PII masking, read-only access).

**Resources provided:** `plant_telemetry.csv` (≈200k readings) and `machines.csv` (asset master). Everything runs locally — but each technique maps directly to its cloud equivalent.

_Phase F — Data Engineering._

#objectives

Speed up queries with indexes and read the query plan

Build a cache-aside layer and measure the hit-rate speedup

Simulate read replicas and observe replication lag

Shard data across nodes and run a scatter-gather query

Add a materialized view and basic security (masking, read-only)

#how to use this lab

Worked demos teach the pattern; 🧪 LAB EXERCISE cells are real tasks — replace `# YOUR CODE HERE`. Run top to bottom with Shift+Enter.

In [1]:
# === SETUP: build the source files if missing ===
import os
import numpy as np
import pandas as pd


def build_plant(tele_path="plant_telemetry.csv", mach_path="machines.csv", seed=232, verbose=False):
    """A larger industrial-IoT telemetry table + machine master — sized so that indexing,
    caching, sharding and materialized-view effects are actually MEASURABLE (U23 Part 2).

      machines.csv          30 machines (asset master)
      plant_telemetry.csv   ~200k sensor readings (the high-volume table)
    """
    rng = np.random.default_rng(seed)

    N_MACH = 30
    machines = pd.DataFrame({
        "machine_id": [f"M{i:03d}" for i in range(1, N_MACH + 1)],
        "line": rng.choice(["LINE_1", "LINE_2", "LINE_3"], N_MACH),
        "machine_type": rng.choice(["pump", "compressor", "conveyor", "press"], N_MACH),
        "install_year": rng.integers(2014, 2023, N_MACH),
        "criticality": rng.choice(["low", "medium", "high"], N_MACH, p=[0.4, 0.4, 0.2]),
    })
    machines.to_csv(mach_path, index=False)

    N = 200_000
    mids = machines.machine_id.values
    metric = rng.choice(["temp", "vibration", "pressure", "current"], N, p=[0.3, 0.3, 0.2, 0.2])
    base = {"temp": 68, "vibration": 2.6, "pressure": 6.2, "current": 31}
    sd = {"temp": 7, "vibration": 0.9, "pressure": 0.7, "current": 4}
    value = np.array([rng.normal(base[m], sd[m]) for m in metric]).round(3)
    ts = pd.Timestamp("2024-06-01") + pd.to_timedelta(np.sort(rng.uniform(0, 14 * 86400, N)), unit="s")
    tele = pd.DataFrame({
        "reading_id": np.arange(N),
        "ts": ts.round("s"),
        "machine_id": rng.choice(mids, N),
        "metric": metric,
        "value": value,
        "quality_flag": rng.choice(["ok", "suspect"], N, p=[0.97, 0.03]),
    })
    tele.to_csv(tele_path, index=False)

    if verbose:
        print("machines:", machines.shape, "| telemetry:", tele.shape)
        print("distinct machines:", tele.machine_id.nunique(), "| metrics:", tele.metric.nunique())
        print("date span:", tele.ts.min(), "->", tele.ts.max())
    return machines, tele

if not (os.path.exists('plant_telemetry.csv') and os.path.exists('machines.csv')):
    build_plant(); print('Generated source files.')
else:
    print('Found the provided source files.')

Generated source files.


In [2]:
import pandas as pd, numpy as np, sqlite3, time, hashlib
tele = pd.read_csv('plant_telemetry.csv', parse_dates=['ts'])
mach = pd.read_csv('machines.csv')
print('telemetry:', tele.shape, '| machines:', mach.shape)
# load into a single SQLite 'database'
DB = 'plant.db'
if os.path.exists(DB): os.remove(DB)
con = sqlite3.connect(DB)
tele.assign(ts=tele.ts.astype(str)).to_sql('telemetry', con, index=False)
mach.to_sql('machines', con, index=False)
con.commit()
print('loaded into SQLite.')

def timed(sql, params=(), repeats=5):
    best = min(_t(sql, params) for _ in range(repeats)); return best
def _t(sql, params):
    t = time.perf_counter(); con.execute(sql, params).fetchall(); return (time.perf_counter()-t)*1000

telemetry: (200000, 6) | machines: (30, 5)
loaded into SQLite.


#1. Indexing & query plans

In [3]:
# -----------------------------------------------------------
# 🔹 1A. A FILTER QUERY: scan vs index
# -----------------------------------------------------------
q = "SELECT * FROM telemetry WHERE machine_id = ? AND metric = ?"
print('plan BEFORE:', con.execute('EXPLAIN QUERY PLAN ' + q, ('M007', 'temp')).fetchall()[0][-1])
t0 = timed(q, ('M007', 'temp'))
con.execute('CREATE INDEX idx_mach_metric ON telemetry(machine_id, metric)'); con.commit()
print('plan AFTER :', con.execute('EXPLAIN QUERY PLAN ' + q, ('M007', 'temp')).fetchall()[0][-1])
t1 = timed(q, ('M007', 'temp'))
print(f'query: {t0:.2f} ms (scan) -> {t1:.2f} ms (indexed) = {t0/max(t1,1e-6):.1f}x faster')

plan BEFORE: SCAN telemetry
plan AFTER : SEARCH telemetry USING INDEX idx_mach_metric (machine_id=? AND metric=?)
query: 22.83 ms (scan) -> 6.56 ms (indexed) = 3.5x faster


#### 🧪 EXERCISE 1 — A covering index
A *covering* index contains every column a query needs, so the database never touches the table.
1. Write a query that selects only `value` filtered by `machine_id` and `metric`. Time it with the existing index.
2. Create an index on `(machine_id, metric, value)` and re-time — the plan should say it uses the index without a table lookup.
3. In a comment, explain the trade-off (indexes speed reads but cost write time and storage).

In [4]:
q_value = "SELECT value FROM telemetry WHERE machine_id = ? AND metric = ?"

# 1. Time with existing index
print('plan BEFORE covering index:', con.execute('EXPLAIN QUERY PLAN ' + q_value, ('M007', 'temp')).fetchall()[0][-1])
t_existing_idx = timed(q_value, ('M007', 'temp'))

# 2. Create covering index and re-time
con.execute('CREATE INDEX idx_mach_metric_value ON telemetry(machine_id, metric, value)'); con.commit()
print('plan AFTER covering index :', con.execute('EXPLAIN QUERY PLAN ' + q_value, ('M007', 'temp')).fetchall()[0][-1])
t_covering_idx = timed(q_value, ('M007', 'temp'))

print(f'Query (value only): {t_existing_idx:.2f} ms (partial index) -> {t_covering_idx:.2f} ms (covering index) = {t_existing_idx/max(t_covering_idx,1e-6):.1f}x faster')

# 3. index trade-off:
# Indexes speed up read queries by allowing the database to quickly locate data without scanning the entire table.
# However, they come with trade-offs: they consume additional storage space, and they slow down write operations
# (INSERT, UPDATE, DELETE) because the index itself must also be updated. The more indexes you have, the higher the write overhead.

plan BEFORE covering index: SEARCH telemetry USING INDEX idx_mach_metric (machine_id=? AND metric=?)
plan AFTER covering index : SEARCH telemetry USING COVERING INDEX idx_mach_metric_value (machine_id=? AND metric=?)
Query (value only): 4.07 ms (partial index) -> 1.17 ms (covering index) = 3.5x faster


#2. Cache-aside caching

In [5]:
# -----------------------------------------------------------
# 🔹 2A. A CACHE-ASIDE LAYER in front of an expensive query
# -----------------------------------------------------------
def expensive_query(machine_id):
    # an aggregation the app asks for repeatedly
    sql = 'SELECT metric, AVG(value) FROM telemetry WHERE machine_id=? GROUP BY metric'
    return tuple(con.execute(sql, (machine_id,)).fetchall())

cache = {}; hits = misses = 0
def get_machine_avgs(machine_id):
    global hits, misses
    if machine_id in cache:           # 1) check cache
        hits += 1; return cache[machine_id]
    misses += 1                        # 2) miss -> hit the DB
    result = expensive_query(machine_id)
    cache[machine_id] = result         # 3) fill the cache
    return result

import random
queries = [f'M{random.randint(1,30):03d}' for _ in range(2000)]   # repeated access pattern
t = time.perf_counter()
for mid in queries: get_machine_avgs(mid)
elapsed = (time.perf_counter()-t)*1000
print(f'2000 lookups in {elapsed:.0f} ms | hits={hits} misses={misses} | hit-rate={hits/2000:.1%}')
print('Only the first lookup per machine touches the DB; the rest are served from memory.')

2000 lookups in 37 ms | hits=1970 misses=30 | hit-rate=98.5%
Only the first lookup per machine touches the DB; the rest are served from memory.


#### 🧪 EXERCISE 2 — Invalidation & speedup
1. Measure the average time of a **cache hit** vs a **cache miss** (a miss runs `expensive_query`). Report the speedup.
2. Cache **invalidation**: write `record_reading(machine_id, ...)` that inserts a row AND evicts that machine's cache entry (so the next read recomputes). Show the entry is gone after a write.
3. In a comment, explain why stale caches are the classic caching bug, and what a TTL would add.

In [6]:
# 1-2. hit vs miss timing; write-with-invalidation

# Part 1: Measure cache hit vs. cache miss speedup
cache.clear() # Clear the cache for a fresh measurement
hits = 0
misses = 0

test_machine_id = 'M001' # Choose an existing machine_id for demonstration

# Time a cache miss (first call to get data for test_machine_id)
t_start_miss = time.perf_counter()
_ = get_machine_avgs(test_machine_id)
t_miss = (time.perf_counter() - t_start_miss) * 1000
print(f'Cache miss for {test_machine_id}: {t_miss:.2f} ms')

# Time a cache hit (second call for the same test_machine_id, data should be cached)
t_start_hit = time.perf_counter()
_ = get_machine_avgs(test_machine_id)
t_hit = (time.perf_counter() - t_start_hit) * 1000
print(f'Cache hit for {test_machine_id}: {t_hit:.2f} ms')
print(f'Cache hit is {t_miss/max(t_hit, 1e-6):.1f}x faster than a miss.\n')


# Part 2: Cache invalidation with record_reading
def record_reading(machine_id, metric, value, quality_flag='ok'):
    global cache
    # Get the current maximum reading_id to generate a new one. In a production system,
    # this would typically use an auto-incrementing primary key or a distributed ID generator.
    max_reading_id_query = con.execute('SELECT MAX(reading_id) FROM telemetry').fetchone()
    max_reading_id = max_reading_id_query[0] if max_reading_id_query[0] is not None else -1
    new_reading_id = max_reading_id + 1

    # Get current timestamp in a format consistent with existing 'ts' column
    new_ts = pd.Timestamp.now().isoformat(timespec='seconds')

    # Insert new record into the database
    con.execute(
        'INSERT INTO telemetry (reading_id, ts, machine_id, metric, value, quality_flag) VALUES (?, ?, ?, ?, ?, ?)',
        (new_reading_id, new_ts, machine_id, metric, value, quality_flag)
    )
    con.commit()
    print(f'New reading for {machine_id} inserted into DB: (ID:{new_reading_id}, TS:{new_ts}, Metric:{metric}, Value:{value}, Quality:{quality_flag})')

    # Invalidate cache entry for this machine_id
    if machine_id in cache:
        del cache[machine_id]
        print(f'Cache entry for {machine_id} invalidated.')
    else:
        print(f'No cache entry for {machine_id} found in cache to invalidate.')

# Demonstrate cache invalidation
print('\n--- Demonstrating Cache Invalidation ---')
cache.clear() # Ensure a clean slate for the demo
hits = 0
misses = 0

demo_machine_id = 'M002'

# Populate cache for demo_machine_id
_ = get_machine_avgs(demo_machine_id)
print(f'Is {demo_machine_id} in cache before write: {demo_machine_id in cache}')

# Record a new reading for demo_machine_id, which should invalidate its cache entry
record_reading(demo_machine_id, 'vibration', 3.5, 'ok')
print(f'Is {demo_machine_id} in cache after write and invalidation: {demo_machine_id in cache}')

# The next call to get_machine_avgs for demo_machine_id should now be a miss again
_ = get_machine_avgs(demo_machine_id)
print(f'Is {demo_machine_id} in cache after re-populating (miss): {demo_machine_id in cache}')
print(f'Current misses: {misses}, hits: {hits}')

# 3. cache staleness & TTL:
# Cache staleness occurs when data in the cache becomes outdated because the underlying source of truth (e.g., database) has changed,
# but the cached copy has not been updated or evicted. This can lead to users seeing inconsistent or incorrect information.
# A Time-To-Live (TTL) mechanism is a common strategy to mitigate staleness. It assigns an expiration time to each item in the cache.
# Once an item's TTL expires, it is automatically removed from the cache, forcing the system to fetch the fresh data from the source on the next request.
# While TTL does not guarantee immediate consistency (data can still be stale until the TTL expires), it ensures eventual consistency within a bounded time frame,
# reducing the likelihood of long-term staleness issues. It also simplifies cache management compared to explicit invalidation for every possible data change.

Cache miss for M001: 2.34 ms
Cache hit for M001: 0.11 ms
Cache hit is 21.5x faster than a miss.


--- Demonstrating Cache Invalidation ---
Is M002 in cache before write: True
New reading for M002 inserted into DB: (ID:200000, TS:2026-06-22T05:40:43, Metric:vibration, Value:3.5, Quality:ok)
Cache entry for M002 invalidated.
Is M002 in cache after write and invalidation: False
Is M002 in cache after re-populating (miss): True
Current misses: 2, hits: 0


#3. Read replicas & replication lag

In [7]:
# -----------------------------------------------------------
# 🔹 3A. PRIMARY takes writes; a REPLICA serves reads (with lag)
# -----------------------------------------------------------
primary = sqlite3.connect('primary.db'); primary.execute('DROP TABLE IF EXISTS kv')
primary.execute('CREATE TABLE kv (machine_id TEXT PRIMARY KEY, status TEXT)')
primary.execute("INSERT INTO kv VALUES ('M007','ok')"); primary.commit()

def make_replica(src='primary.db', dst='replica.db'):
    import shutil; shutil.copy(src, dst); return sqlite3.connect(dst)
replica = make_replica()   # snapshot copy = a (lagging) read replica

# a WRITE goes to the primary only
primary.execute("UPDATE kv SET status='fault' WHERE machine_id='M007'"); primary.commit()
p = primary.execute("SELECT status FROM kv WHERE machine_id='M007'").fetchone()[0]
r = replica.execute("SELECT status FROM kv WHERE machine_id='M007'").fetchone()[0]
print(f'read from PRIMARY: {p}  |  read from REPLICA: {r}  <- stale (replication lag)')
replica = make_replica()   # replication catches up (re-sync)
r2 = replica.execute("SELECT status FROM kv WHERE machine_id='M007'").fetchone()[0]
print(f'after replica re-syncs: {r2}  (eventual consistency)')

read from PRIMARY: fault  |  read from REPLICA: ok  <- stale (replication lag)
after replica re-syncs: fault  (eventual consistency)


#### 🧪 EXERCISE 3 — Read/write routing
1. Write a tiny router: `def query(sql, write=False)` that sends writes to `primary` and reads to `replica`. Use it to do one write then one read.
2. In a comment, explain when routing reads to replicas is safe (analytics, dashboards) and when it is dangerous (read-after-write where the user expects to see their own change immediately).

In [10]:
# 1. read/write router
def query(sql, params=(), write=False):
    if write:
        # Writes go to the primary
        conn = primary
        result = conn.execute(sql, params)
        conn.commit()
        print(f"Write executed on primary: {sql} with {params}")
    else:
        # Reads go to the replica
        conn = replica
        result = conn.execute(sql, params)
        print(f"Read executed on replica: {sql} with {params}")
    return result

# Demonstrate usage:
print('\n--- Demonstrating Read/Write Routing ---')

# Ensure the primary is updated for demonstration
primary.execute("INSERT OR REPLACE INTO kv VALUES (?,?)", ('M001', 'healthy'))
primary.commit()
replica = make_replica() # Resync replica for current state

# Perform a write operation (e.g., update M001 status to 'critical')
query("UPDATE kv SET status=? WHERE machine_id=?", ('critical', 'M001'), write=True)

# Perform a read operation (e.g., get status for M001)
# Note: This read will hit the (potentially stale) replica
status_from_replica = query("SELECT status FROM kv WHERE machine_id=?", ('M001',)).fetchone()[0]
print(f"Status of M001 from replica: {status_from_replica}")

# To see the updated status from the primary for comparison
status_from_primary = primary.execute("SELECT status FROM kv WHERE machine_id=?", ('M001',)).fetchone()[0]
print(f"Status of M001 from primary: {status_from_primary}")

# Re-sync the replica to show eventual consistency
replica = make_replica() # Replica catches up to primary
status_after_resync = query("SELECT status FROM kv WHERE machine_id=?", ('M001',)).fetchone()[0]
print(f"Status of M001 from replica after resync: {status_after_resync}")

# 2. when replica reads are (un)safe:
# Routing reads to replicas is safe for applications that can tolerate some degree of data staleness, such as analytics dashboards,
# reporting tools, or non-critical background processes. These scenarios benefit from reduced load on the primary database
# and improved read performance without requiring immediate consistency.
#
# However, it is dangerous for scenarios that require strong consistency, especially "read-after-write" operations where a user
# expects to see their own changes immediately. If a user performs a write (e.g., updates their profile) and then immediately
# attempts to read that updated information from a lagging replica, they might see the old, stale data. This can lead to a poor
# user experience or even application logic errors if the application assumes immediate consistency.


--- Demonstrating Read/Write Routing ---
Write executed on primary: UPDATE kv SET status=? WHERE machine_id=? with ('critical', 'M001')
Read executed on replica: SELECT status FROM kv WHERE machine_id=? with ('M001',)
Status of M001 from replica: healthy
Status of M001 from primary: critical
Read executed on replica: SELECT status FROM kv WHERE machine_id=? with ('M001',)
Status of M001 from replica after resync: critical


#4. Sharding across nodes

In [11]:
# -----------------------------------------------------------
# 🔹 4A. SHARD the telemetry across N nodes by machine_id
# -----------------------------------------------------------
import zlib
N_SHARDS = 4
def shard_of(machine_id): return zlib.crc32(machine_id.encode()) % N_SHARDS

shards = [sqlite3.connect(f'shard_{i}.db') for i in range(N_SHARDS)]
for sc in shards:
    sc.execute('DROP TABLE IF EXISTS telemetry')
    sc.execute('CREATE TABLE telemetry (machine_id TEXT, metric TEXT, value REAL)')
rows = con.execute('SELECT machine_id, metric, value FROM telemetry').fetchall()
for mid, metric, val in rows:
    shards[shard_of(mid)].execute('INSERT INTO telemetry VALUES (?,?,?)', (mid, metric, val))
for sc in shards: sc.commit()
sizes = [sc.execute('SELECT COUNT(*) FROM telemetry').fetchone()[0] for sc in shards]
print('rows per shard:', sizes, '-> total', sum(sizes))

# a single-machine query hits ONE shard (routed); a global query is scatter-gather
one = shards[shard_of('M007')].execute("SELECT COUNT(*) FROM telemetry WHERE machine_id='M007'").fetchone()[0]
print('M007 lives on shard', shard_of('M007'), 'with', one, 'rows (routed read, no fan-out)')

rows per shard: [53096, 53359, 46973, 46573] -> total 200001
M007 lives on shard 0 with 6687 rows (routed read, no fan-out)


#### 🧪 EXERCISE 4 — Scatter-gather & skew
1. Compute the **global** average `value` per metric by querying **every** shard and combining the partial results (scatter-gather). Compare to the single-DB answer.
2. Show the downside of a bad shard key: count rows if you sharded by **metric** (only 4 distinct values) instead of machine_id — note how few shards would be used and why that causes hotspots.
3. In a comment, state the rule for picking a shard key (high cardinality, even access).

In [12]:
# 1-2. scatter-gather global average; metric-skew demonstration

# Part 1: Compute global average value per metric using scatter-gather
print('--- Scatter-Gather Global Average (value per metric) ---')
combined_results = []
for i, shard_conn in enumerate(shards):
    # Query each shard for average value per metric
    shard_avg = shard_conn.execute('SELECT metric, AVG(value) FROM telemetry GROUP BY metric').fetchall()
    combined_results.extend(shard_avg)
    # print(f'Shard {i} partial results: {shard_avg}')

# Combine and aggregate results from all shards
# Convert to DataFrame for easy aggregation
df_combined = pd.DataFrame(combined_results, columns=['metric', 'avg_value'])
global_avg_per_metric = df_combined.groupby('metric')['avg_value'].mean().reset_index()
print('\nGlobal Average (Scatter-Gather):')
print(global_avg_per_metric)

# Compare to single-DB answer
single_db_avg = con.execute('SELECT metric, AVG(value) FROM telemetry GROUP BY metric').fetchall()
df_single_db = pd.DataFrame(single_db_avg, columns=['metric', 'avg_value_single_db'])
print('\nSingle-DB Average:')
print(df_single_db)

print('\nComparison (should be very close, differences due to floating point):')
merged_avgs = pd.merge(global_avg_per_metric, df_single_db, on='metric')
merged_avgs['diff'] = (merged_avgs['avg_value'] - merged_avgs['avg_value_single_db']).abs()
print(merged_avgs)

# Part 2: Show the downside of a bad shard key (sharding by metric)
print('\n--- Demonstrating Sharding Skew by Metric ---')

N_METRIC_SHARDS = len(tele.metric.unique()) # Number of unique metrics
def shard_of_metric(metric_name): # A shard key based on metric
    # Use a simple hash or ordinal mapping for demonstration
    return zlib.crc32(metric_name.encode()) % N_METRIC_SHARDS

# Create new shards for metrics
shards_metric = [sqlite3.connect(f'shard_metric_{i}.db') for i in range(N_METRIC_SHARDS)]
for sm_conn in shards_metric:
    sm_conn.execute('DROP TABLE IF EXISTS telemetry')
    sm_conn.execute('CREATE TABLE telemetry (machine_id TEXT, metric TEXT, value REAL)')

# Distribute data by metric
for mid, metric, val in con.execute('SELECT machine_id, metric, value FROM telemetry').fetchall():
    shards_metric[shard_of_metric(metric)].execute('INSERT INTO telemetry VALUES (?,?,?)', (mid, metric, val))
for sm_conn in shards_metric: sm_conn.commit()

sizes_metric = [sm_conn.execute('SELECT COUNT(*) FROM telemetry').fetchone()[0] for sm_conn in shards_metric]
print('Rows per shard when sharding by metric (skewed):', sizes_metric)
print('This demonstrates that some shards receive significantly more data than others, creating hotspots and underutilized shards.')

# 3. shard-key rule:
# A good shard key should have high cardinality (many distinct values) to distribute data evenly across shards and prevent hot-spots.
# It should also ensure even access patterns, meaning queries are distributed fairly across shards, minimizing the chance of one shard becoming a bottleneck.
# Ideal shard keys are often derived from entity identifiers (like user_id, machine_id) that are frequently queried and have a uniform distribution.

--- Scatter-Gather Global Average (value per metric) ---

Global Average (Scatter-Gather):
      metric  avg_value
0    current  30.973752
1   pressure   6.195560
2       temp  68.005096
3  vibration   2.598349

Single-DB Average:
      metric  avg_value_single_db
0    current            30.974039
1   pressure             6.195500
2       temp            68.005528
3  vibration             2.598373

Comparison (should be very close, differences due to floating point):
      metric  avg_value  avg_value_single_db      diff
0    current  30.973752            30.974039  0.000287
1   pressure   6.195560             6.195500  0.000060
2       temp  68.005096            68.005528  0.000431
3  vibration   2.598349             2.598373  0.000023

--- Demonstrating Sharding Skew by Metric ---
Rows per shard when sharding by metric (skewed): [0, 59811, 60033, 80157]
This demonstrates that some shards receive significantly more data than others, creating hotspots and underutilized shards.


#5. Materialized view & security

In [13]:
# -----------------------------------------------------------
# 🔹 5A. MATERIALIZED VIEW: precompute an expensive rollup
# -----------------------------------------------------------
rollup_sql = '''SELECT machine_id, metric, AVG(value) avg_v, COUNT(*) n
                FROM telemetry GROUP BY machine_id, metric'''
t_live = timed(rollup_sql)                       # computed on the fly every time
con.execute('DROP TABLE IF EXISTS mv_machine_metric')
con.execute('CREATE TABLE mv_machine_metric AS ' + rollup_sql); con.commit()
con.execute('CREATE INDEX idx_mv ON mv_machine_metric(machine_id)'); con.commit()
t_mv = timed('SELECT * FROM mv_machine_metric WHERE machine_id=?', ('M007',))
print(f'aggregation live: {t_live:.2f} ms  |  read from materialized view: {t_mv:.2f} ms')
print('Trade-off: the view must be REFRESHED when underlying data changes (it can go stale).')

aggregation live: 34.70 ms  |  read from materialized view: 0.01 ms
Trade-off: the view must be REFRESHED when underlying data changes (it can go stale).


In [14]:
# -----------------------------------------------------------
# 🔹 5B. SECURITY: mask PII-like fields & a read-only connection
# -----------------------------------------------------------
# tokenize an identifier so analysts can join without seeing the raw id
def tokenize(s, salt='plant2024'):
    return hashlib.sha256((salt + str(s)).encode()).hexdigest()[:12]
print('M007 tokenized ->', tokenize('M007'))
# a read-only connection: writes are rejected (least privilege)
ro = sqlite3.connect('file:plant.db?mode=ro', uri=True)
try:
    ro.execute('DELETE FROM telemetry'); print('write succeeded (unexpected!)')
except Exception as e:
    print('write blocked on read-only connection:', type(e).__name__)

M007 tokenized -> 936b3bb31c50
write blocked on read-only connection: OperationalError


#### 🧪 EXERCISE 5 — Keyset pagination
Offset pagination (`LIMIT n OFFSET k`) gets slower as `k` grows because the DB still scans the skipped rows.
1. Time `... ORDER BY reading_id LIMIT 50 OFFSET 150000` vs **keyset** pagination `... WHERE reading_id > 150000 ORDER BY reading_id LIMIT 50`.
2. In a comment, explain why keyset pagination stays fast on deep pages (it seeks via the index instead of counting past skipped rows).

In [15]:
# 1. offset vs keyset pagination timing
OFFSET_VAL = 150000
LIMIT_VAL = 50

# Offset pagination query
q_offset = f"SELECT reading_id FROM telemetry ORDER BY reading_id LIMIT {LIMIT_VAL} OFFSET {OFFSET_VAL}"
print(f'Offset pagination query plan: {con.execute('EXPLAIN QUERY PLAN ' + q_offset).fetchall()[0][-1]}')
t_offset = timed(q_offset)
print(f'Offset pagination (OFFSET {OFFSET_VAL}): {t_offset:.2f} ms')

# Keyset pagination query
# Assuming reading_id is a unique, indexed, and monotonically increasing column
q_keyset = f"SELECT reading_id FROM telemetry WHERE reading_id > {OFFSET_VAL} ORDER BY reading_id LIMIT {LIMIT_VAL}"
print(f'Keyset pagination query plan: {con.execute('EXPLAIN QUERY PLAN ' + q_keyset).fetchall()[0][-1]}')
t_keyset = timed(q_keyset)
print(f'Keyset pagination (WHERE reading_id > {OFFSET_VAL}): {t_keyset:.2f} ms')

print(f'\nKeyset pagination is {t_offset/max(t_keyset, 1e-6):.1f}x faster than offset pagination for deep pages.')

# 2. why keyset wins on deep pages:
# Offset pagination (LIMIT n OFFSET k) becomes progressively slower for larger 'k' values (deeper pages).
# This is because the database still has to scan or traverse 'k' rows before it can start returning the 'n' requested rows.
# Even if an index is used for ordering, the database often has to step through the index entries one by one until it reaches the starting point 'k'.
#
# Keyset pagination (WHERE primary_key > last_seen_id ORDER BY primary_key LIMIT n) avoids this inefficiency.
# By using a WHERE clause on an indexed column (like a primary key or unique identifier), the database can directly seek to the starting point
# in the index, completely skipping the 'k' rows that precede it. It only reads 'n' rows starting from that seek point.
# This means the performance of keyset pagination is largely independent of the page depth, making it consistently fast even for very deep pages, as long as the filtering column is indexed.

Offset pagination query plan: SCAN telemetry
Offset pagination (OFFSET 150000): 74.24 ms
Keyset pagination query plan: SCAN telemetry
Keyset pagination (WHERE reading_id > 150000): 15.67 ms

Keyset pagination is 4.7x faster than offset pagination for deep pages.


#📘 Summary

| Technique | What you measured |
| --------- | ----------------- |
| Indexing | scan → index search, large speedup |
| Cache-aside | repeated reads served from memory |
| Read replicas | offload reads; accept replication lag |
| Sharding | spread data by key; route or scatter-gather |
| Materialized view | precompute rollups; refresh to stay fresh |
| Security | tokenize PII; read-only least-privilege access |

**Core lesson:** production performance comes from a toolbox of trade-offs — cache hot reads, index what you filter, replicate and shard to scale out, precompute expensive rollups — each buying speed or scale at the cost of freshness, complexity or write overhead.

**Next — U24:** apply this same engineering discipline to machine-learning systems (MLOps).